In [0]:
#1 Importing Tools 
import openpyxl
import pandas as pd

from pyspark.sql import functions as F
from datetime import datetime
from openpyxl.styles import NamedStyle

In [0]:
#2 Reduce risk of a timeout by increasing limit to 30 minutes
spark.conf.set("spark.databricks.execution.timeout", "1800")

In [0]:
#3 Loading the master hierarchies table from the lake mart
#df_master_hierarchies = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/#EROC_Collection_Queries/master_hierarchies_table.csv")

df_master_hierarchies = spark.table ("udal_gold_restricted.ukhd_ods.provider_hierarchies_icb")

display(df_master_hierarchies.limit(10))
print(f"Number of rows in master hierarchies: {df_master_hierarchies.filter(F.col('Effective_To').isNull()).count()}")



In [0]:
#4 loading ICB to Region mapping table
df_icb_region = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_ICB_Region_DisplayNames.csv")  

#display(df_icb_region.limit(10))
#print(f"Number of rows in icb_region: {df_icb_region.count()}")

In [0]:
#5 loading list of merged providers
df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

display(df_merged_providers.limit(100))
#print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#6 creating new provider code from the provider mapping table
provider_code_mapping = df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

#display(df_merged_providers.limit(10))
#print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#7a importing MHS metric list and internal ID
mhs_metric_list = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS")

#display(mhs_metric_list.limit(10))
#print(f"Number of rows in mhs_metric_list: {mhs_metric_list.count()}")

#7b importing MHS allowable org codes
mhs_allowable_orgs = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/Allowable_Org_Codes_Status.csv")


#display(mhs_allowable_orgs.limit(10))
#print(f"Number of rows in mhs_allowable_orgs: {mhs_allowable_orgs.count()}")

In [0]:
#8 Loading the core monthly snapshot data
from pyspark.sql import functions as F
df_op_activity_snapshot = spark.table('udal_gold_restricted.mesh_opa.opa_core_monthly_snapshot')

display(df_op_activity_snapshot.limit(10))

# Show number of rows in the raw data
row_count = df_op_activity_snapshot.count()
print(f"Number of rows in raw data: {row_count}")

In [0]:
#9 Creating the wide table & inserting new column for merged providers
# with new merger codes and mapping to ICB and Region codes

from pyspark.sql.functions import (
    when,
    col,
    lit,
    create_map,
    coalesce,
    last_day,
    to_date,
    concat_ws
)
import pyspark.sql.functions as F

# ---------------------------------------
# 1. Define valid treatment function codes
# ---------------------------------------

VALID_TREATMENT_CODES = [
    '100', '101', '102', '104', '105', '106', '108', '110', '111', '115', '120', '130', '140',
    '144', '145', '301', '302', '303', '307', '320', '330', '340', '361', '400', '410', '420',
    '430', '501', '502', '560', '650'
]

# ---------------------------------------------------
# 2. Add Treatment_Function_Code_New (valid vs Other)
# ---------------------------------------------------

opa_with_tfc = df_op_activity_snapshot.withColumn(
    "Treatment_Function_Code_New",
    when(col("Treatment_Function_Code").isin(VALID_TREATMENT_CODES),
         col("Treatment_Function_Code")
    ).otherwise("Other")
)

# ---------------------------------------------------
# 3. Add Treatment_Function_Group (GS / OMFS / T&O / code)
# ---------------------------------------------------

opa_with_groups = opa_with_tfc.withColumn(
    "Treatment_Function_Group",
    when(col("Treatment_Function_Code_New").isin("100", "102", "104", "105", "106"), "GS")
    .when(col("Treatment_Function_Code_New").isin("140", "144", "145"), "OMFS")
    .when(col("Treatment_Function_Code_New").isin("110", "111", "115"), "T&O")
    .otherwise(col("Treatment_Function_Code_New"))
)

# ---------------------------------------------------
# 4. Filter dataset (years, admin category, TFC, attendance)
# ---------------------------------------------------

opa_filtered = opa_with_groups.filter(
    (col("Der_Financial_Year").isin("2023/24", "2024/25", "2025/26")) &  # Update manually if needed
    (col("Administrative_Category") == "01") &
    (col("Treatment_Function_Code") != "812") &
    (col("First_Attendance").isin("1", "2", "3", "4"))
)

# ---------------------------------------------------
# 5. Aggregate metrics by month, provider, and TFC group
# ---------------------------------------------------

opa_agg = opa_filtered.groupBy(
    "Der_Activity_Month",
    "Der_Provider_Code",
    "Treatment_Function_Group"
).agg(
    # All contacts
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_NoProc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_NoProc"),

    # Face-to-face
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "1"), 1
        ).otherwise(0)
    ).alias("F2F_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "2"), 1
        ).otherwise(0)
    ).alias("F2F_FU"),

    # Remote
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "3"), 1
        ).otherwise(0)
    ).alias("Remote_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "4"), 1
        ).otherwise(0)
    ).alias("Remote_FU"),

    # Did not attends (DNAs)
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_DNA"),

    # 2WW DNA
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW_DNA"),

    # All 2WW appointments
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW")
)

# ---------------------------------------------------
# 6. Add "All" TFC totals by month and provider
# ---------------------------------------------------

METRIC_COLS = [
    c for c in opa_agg.columns
    if c not in ["Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group"]
]

opa_all_tfc = opa_agg.groupBy("Der_Activity_Month", "Der_Provider_Code").agg(
    *[F.sum(col(c)).alias(c) for c in METRIC_COLS]
).withColumn("Treatment_Function_Group", lit("All"))

opa_final = opa_agg.unionByName(opa_all_tfc)

# Order by results
opa_final_ordered = opa_final.orderBy("Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group")

# ---------------------------------------------------
# 7. Normalise provider codes -> Provider_Base_Code
#     (replicates SQL CASE logic)
# ---------------------------------------------------

from pyspark.sql.functions import when, col

opa_final_ordered = opa_final_ordered.withColumn(
    "Provider_Base_Code",
    when((
        col("Der_Provider_Code").substr(4, 2) == "00")&(col("Der_Provider_Code")!="NQT00"),   # e.g. ABC00
        col("Der_Provider_Code").substr(1, 3)            # -> ABC
    ).otherwise(
        col("Der_Provider_Code")                         # everything else unchanged
    )
)

# ---------------------------------------------------
# 8. Build mapping from df_merged_providers (Old -> New)
# ---------------------------------------------------

provider_code_mapping_dict = {
    row["Old_Provider_Code"]: row["New_Provider_Code"]
    for row in df_merged_providers.select("Old_Provider_Code", "New_Provider_Code").distinct().collect()
}

mapping_list = []
for k, v in provider_code_mapping_dict.items():
    mapping_list.append(lit(k))
    mapping_list.append(lit(v))

# IMPORTANT: use *mapping_list, not mapping_list
mapping_expr = create_map(*mapping_list)

# ---------------------------------------------------
# 9. Add "Adj Org Code" based on Provider_Base_Code & mapping
# ---------------------------------------------------

opa_final_ordered_with_adj = opa_final_ordered.withColumn(
    "Adj Org Code",
    coalesce(mapping_expr.getItem(col("Provider_Base_Code")), col("Provider_Base_Code"))
)

# ---------------------------------------------------
# 10. Build Org_Code_For_Join to match hierarchy:
#       - If 5-char and ends with "00" -> first 3 chars
#       - Else use the full Adj Org Code
#     This reproduces LEFT(Org,3) for trust codes.
# ---------------------------------------------------

opa_with_join_code = opa_final_ordered_with_adj.withColumn(
    "Org_Code_For_Join",
    when(
        (F.length(col("Adj Org Code")) == 5) & col("Adj Org Code").endswith("00"),
        col("Adj Org Code").substr(1, 3)
    ).otherwise(col("Adj Org Code"))
)

# ---------------------------------------------------
# 11. Join to master hierarchies to get ICB (ICB_Code)
# ---------------------------------------------------

opa_with_icb = opa_with_join_code.join(
    df_master_hierarchies.select(
        F.col("Organisation_Code").alias("join_org_code"),
        F.col("ICB_Code").alias("ICB")
    ),
    opa_with_join_code["Org_Code_For_Join"] == F.col("join_org_code"),
    "left"
).drop("join_org_code")

# ---------------------------------------------------
# 12. Join to df_icb_region to get Region_Code
# ---------------------------------------------------

opa_with_icb_region = opa_with_icb.join(
    df_icb_region.select(
        F.col("ICB_Code").alias("join_icb"),
        F.col("Region_Code")
    ),
    opa_with_icb["ICB"] == F.col("join_icb"),
    "left"
).drop("join_icb")

# ---------------------------------------------------
# 13. Build Der_Activity_Month_Date (YYYYMM -> month end date)
# ---------------------------------------------------

opa_with_icb_region = opa_with_icb_region.withColumn(
    "Der_Activity_Month_Date",
    last_day(
        to_date(
            concat_ws(
                "-",
                col("Der_Activity_Month").substr(1, 4),
                col("Der_Activity_Month").substr(5, 2),
                lit("01")
            )
        )
    )
)

# ---------------------------------------------------
# 14. Aggregate to final level & sort
# ---------------------------------------------------

# Identifier columns to keep
id_cols = ["Der_Activity_Month_Date", "Treatment_Function_Group", "Region_Code", "ICB", "Adj Org Code"]

# Metric columns = everything else except IDs & things we don't want
metric_cols = [
    c for c in opa_with_icb_region.columns
    if c not in id_cols + [
        "Der_Activity_Month",
        "Der_Provider_Code",
        "Treatment_Function_Code_New",
        "Provider_Base_Code",
        "Org_Code_For_Join"
    ]
]

opa_final_processed = (
    opa_with_icb_region
    .groupBy(*[F.col(c) for c in id_cols])
    .agg(*[F.sum(F.col(c)).alias(c) for c in metric_cols])
    .orderBy("Der_Activity_Month_Date", "Region_Code", "ICB", "Adj Org Code", "Treatment_Function_Group")
)

# Optional: quick peek / row count if you want
display(opa_final_processed.limit(10))
print(f"opa_final_processed: {opa_final_processed.count()}")




In [0]:
#10 — Safe metric calculation (robust to missing columns)
from pyspark.sql import functions as F

df = df = opa_final_processed 

def safe_add(df, new_col, expr_fn, required_cols):
    if all(c in df.columns for c in required_cols):
        return df.withColumn(new_col, expr_fn(df))
    else:
        return df.withColumn(new_col, F.lit(None))

metrics = [
    ("All_DNA_Over_All_Total", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_DNA_Over_All_Total_IG", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_First_DNA_Over_All_First", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_First_DNA_Over_All_First_IG", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_FU_DNA_Over_All_FU", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_FU_DNA_Over_All_FU_IG", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_2WW_DNA_Over_All_2WW", lambda d: F.when(
        (F.col("All_2WW") != 0) & (F.col("All_2WW").isNotNull()),
        (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
    ), ["All_2WW_DNA", "All_2WW"]),
    ("All_FU_2WW_DNA_Over_All_FU_2WW", lambda d: F.when(
        (F.col("All_FU_2WW") != 0) & (F.col("All_FU_2WW").isNotNull()),
        (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
    ), ["All_FU_2WW_DNA", "All_FU_2WW"]),
    ("All_First_2WW_DNA_Over_All_First_2WW", lambda d: F.when(
        (F.col("All_First_2WW") != 0) & (F.col("All_First_2WW").isNotNull()),
        (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
    ), ["All_First_2WW_DNA", "All_First_2WW"]),
    ("All_FU_NoProc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
    ), ["All_FU_NoProc", "All_FU"]),
    ("All_FU_Proc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
    ), ["All_FU_Proc", "All_FU"]),
    ("All_FU_To_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
    ), ["All_FU", "All_First"]),
    ("All_FU_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
    ), ["All_FU", "All_Total"]),
    ("All_First_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
    ), ["All_First", "All_Total"]),
    ("All_NoProc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
    ), ["All_NoProc", "All_Total"]),
    ("All_Proc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
    ), ["All_Proc", "All_Total"]),
    ("Remote_Total_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
    ), ["Remote_Total", "All_Total"]),
    ("Remote_FU_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
    ), ["Remote_FU", "All_FU"]),
    ("Remote_First_Over_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
    ), ["Remote_First", "All_First"]),
    ("F2F_DNA_Over_F2F_Total", lambda d: F.when(
        (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
        (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
    ), ["F2F_Total", "F2F_DNA"]),
    ("Remote_DNA_Over_Remote_Total", lambda d: F.when(
        (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
        (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
    ), ["Remote_Total", "Remote_DNA"]),
]

for name, expr, req in metrics:
    df = safe_add(df, name, expr, req)

simple_copies = [
    ("All_DNA1", "All_DNA"),
    ("All_DNA2", "All_DNA"),
    ("All_First1", "All_First"),
    ("All_First2", "All_First"),
    ("All_First3", "All_First"),
    ("All_FU1", "All_FU"),
    ("All_FU2", "All_FU"),
    ("All_FU3", "All_FU"),
    ("All_FU4", "All_FU"),
    ("All_FU5", "All_FU"),
    ("All_Total1", "All_Total"),
    ("All_Total2", "All_Total"),
    ("All_Total3", "All_Total"),
    ("All_Total4", "All_Total"),
    ("All_Total5", "All_Total"),
    ("All_Total6", "All_Total"),
    ("Remote_Total1", "Remote_Total"),
    ("Remote_Total2", "Remote_Total"),
]
for newc, base in simple_copies:
    if base in df.columns:
        df = df.withColumn(newc, F.col(base))
    else:
        df = df.withColumn(newc, F.lit(None))

combos = [
    ("All_First_plus_All_First_DNA", ["All_First", "All_First_DNA"]),
    ("All_FU_plus_All_FU_DNA", ["All_FU", "All_FU_DNA"]),
    ("All_Total_plus_All_DNA", ["All_Total", "All_DNA"]),
    ("F2F_Total_plus_F2F_DNA", ["F2F_Total", "F2F_DNA"]),
    ("Remote_Total_plus_Remote_DNA", ["Remote_Total", "Remote_DNA"]),
]
for newc, cols in combos:
    if all(c in df.columns for c in cols):
        df = df.withColumn(newc, F.col(cols[0]).cast("long") + F.col(cols[1]).cast("long"))
    else:
        df = df.withColumn(newc, F.lit(None))

cols_to_drop = [c for c in ["Der_Activity_Month"] if c in df.columns]
opa_final_with_added_metrics = df.drop(*cols_to_drop)

#opa_final_with_added_metrics.coalesce(1).write \
    #.format("csv") \
    #.mode("overwrite") \
    #.option("header", "true") \
    #.save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/#opa_final_with_added_metrics_full_export.csv")

display(opa_final_with_added_metrics.limit(10))
print(f"opa_final_with_added_metrics: {opa_final_with_added_metrics.count()} rows")

In [0]:
# 11 reshapes the wide outpatient dataset into a long (tidy) format for easier analysis
from pyspark.sql.functions import col, explode, array, struct, lit, concat_ws

# ID columns to keep
id_cols = [
    "Der_Activity_Month_Date",
    "Treatment_Function_Group",
    "Adj Org Code",
    "ICB",
    "Region_Code"
]

# Identify all metric columns
metric_cols = [c for c in opa_final_with_added_metrics.columns if c not in id_cols]

# Unpivot all metrics, casting values to string to avoid type errors
opa_long = (
    opa_final_with_added_metrics
    .select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("Metric_Name"),
                col(c).cast("string").alias("Metric_Value")
            ) for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Create the combined metric name
opa_long = opa_long.withColumn(
    "Metric_Name_Treatment_Function_Group",
    concat_ws("_", col("Metric_Name"), col("Treatment_Function_Group"))
)

# Order by date
opa_long_ordered = opa_long.orderBy("Der_Activity_Month_Date")

display(opa_long_ordered.limit(10))
#print(f"Number of rows in opa_long_ordered: {opa_long_ordered.count()}"))

In [0]:

# Container 12 — Aggregation for Org / ICB / Region (all sectors)


from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit

# ---------------------------------------------------
# 1. Start from Org-level wide dataset from Container 10
# ---------------------------------------------------
df_org = opa_final_with_added_metrics.withColumnRenamed("Adj Org Code", "Adj_Org_Code")

# ---------------------------------------------------
# 2. (Optional) Attach sector type from df_master_hierarchies
#     - Kept for reference / QA, not used in rollups
# ---------------------------------------------------
sector_lookup = (
    df_master_hierarchies
    .select(
        col("Organisation_Code").alias("ODS_Code"),
        col("ODS_Organisation_Type")
    )
    .distinct()
)

df_org = (
    df_org
    .join(
        sector_lookup,
        df_org["Adj_Org_Code"] == col("ODS_Code"),
        "left"
    )
    .drop("ODS_Code")
)

# ---------------------------------------------------
# 3. Metric columns to sum at ICB/Region
# ---------------------------------------------------
BASE_COUNT_COLS = [
    "All_Total","All_First","All_FU","All_Proc","All_NoProc",
    "All_FU_Proc","All_FU_NoProc",
    "F2F_Total","F2F_First","F2F_FU","F2F_DNA",
    "Remote_Total","Remote_First","Remote_FU","Remote_DNA",
    "All_DNA","All_First_DNA","All_FU_DNA",
    "All_2WW","All_First_2WW","All_FU_2WW",
    "All_2WW_DNA","All_First_2WW_DNA","All_FU_2WW_DNA"
]

# Small safety: only sum columns that actually exist
count_cols = [c for c in BASE_COUNT_COLS if c in df_org.columns]

# ---------------------------------------------------
# 4. Function to add rate metrics
# ---------------------------------------------------
def add_rate_metrics(df):
    return (
        df
        # DNA metrics
        .withColumn("All_DNA_Over_All_Total", F.when(
            (F.col("All_Total") + F.col("All_DNA")) != 0,
            (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
        ))
        .withColumn("All_DNA_Over_All_Total_IG", F.col("All_DNA_Over_All_Total"))
        .withColumn("All_First_DNA_Over_All_First", F.when(
            (F.col("All_First") + F.col("All_First_DNA")) != 0,
            (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
        ))
        .withColumn("All_First_DNA_Over_All_First_IG", F.col("All_First_DNA_Over_All_First"))
        .withColumn("All_FU_DNA_Over_All_FU", F.when(
            (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
            (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
        ))
        .withColumn("All_FU_DNA_Over_All_FU_IG", F.col("All_FU_DNA_Over_All_FU"))
        # FU metrics
        .withColumn("All_FU_NoProc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_Proc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_To_All_First", F.when(
            F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
        ))
        # 2WW
        .withColumn("All_2WW_DNA_Over_All_2WW", F.when(
            F.col("All_2WW") != 0, (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
        ))
        .withColumn("All_First_2WW_DNA_Over_All_First_2WW", F.when(
            F.col("All_First_2WW") != 0, (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
        ))
        .withColumn("All_FU_2WW_DNA_Over_All_FU_2WW", F.when(
            F.col("All_FU_2WW") != 0, (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
        ))
        # Mix shares
        .withColumn("All_FU_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
        ))
        .withColumn("All_First_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
        ))
        .withColumn("All_NoProc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
        ))
        .withColumn("All_Proc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_Total_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_FU_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
        ))
        .withColumn("Remote_First_Over_All_First", F.when(
            F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
        ))
        .withColumn("Remote_DNA_Over_Remote_Total", F.when(
            (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
            (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
        ))
        .withColumn("F2F_DNA_Over_F2F_Total", F.when(
            (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
            (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
        ))
    )

# ---------------------------------------------------
# 5. Aggregate to ICB and Region (all orgs included)
# ---------------------------------------------------
df_icb = (
    df_org
    .groupBy("Der_Activity_Month_Date", "ICB", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_icb = add_rate_metrics(df_icb).withColumn("Level", lit("ICB"))

df_region = (
    df_org
    .groupBy("Der_Activity_Month_Date", "Region_Code", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_region = add_rate_metrics(df_region).withColumn("Level", lit("Region"))

# ---------------------------------------------------
# 6. Org-level rows
# ---------------------------------------------------
df_org = df_org.withColumn("Level", lit("Org"))

# ---------------------------------------------------
# 7. Combine all levels
# ---------------------------------------------------
final_output = (
    df_org.unionByName(df_icb, allowMissingColumns=True)
          .unionByName(df_region, allowMissingColumns=True)
)

# ---------------------------------------------------
# 8. Final code column
# ---------------------------------------------------
final_output = final_output.withColumn(
    "Adj_Org_Code_Final",
    when(col("Level") == "Org", col("Adj_Org_Code"))
    .when(col("Level") == "ICB", col("ICB"))
    .when(col("Level") == "Region", col("Region_Code"))
)

# ---------------------------------------------------
# 9. Simple-copy and combo columns
# ---------------------------------------------------
copy_map = {
    "All_DNA1": "All_DNA",
    "All_DNA2": "All_DNA",
    "All_FU1": "All_FU",
    "All_FU2": "All_FU",
    "All_FU3": "All_FU",
    "All_FU4": "All_FU",
    "All_FU5": "All_FU",
    "All_First1": "All_First",
    "All_First2": "All_First",
    "All_First3": "All_First",
    "All_Total1": "All_Total",
    "All_Total2": "All_Total",
    "All_Total3": "All_Total",
    "All_Total4": "All_Total",
    "All_Total5": "All_Total",
    "All_Total6": "All_Total",
    "Remote_Total1": "Remote_Total",
    "Remote_Total2": "Remote_Total"
}

for newc, base in copy_map.items():
    if base in final_output.columns:
        final_output = final_output.withColumn(newc, col(base))

combo_pairs = {
    "All_First_plus_All_First_DNA": ("All_First", "All_First_DNA"),
    "All_FU_plus_All_FU_DNA": ("All_FU", "All_FU_DNA"),
    "All_Total_plus_All_DNA": ("All_Total", "All_DNA"),
    "F2F_Total_plus_F2F_DNA": ("F2F_Total", "F2F_DNA"),
    "Remote_Total_plus_Remote_DNA": ("Remote_Total", "Remote_DNA")
}

for newc, (a, b) in combo_pairs.items():
    if a in final_output.columns and b in final_output.columns:
        final_output = final_output.withColumn(newc, col(a).cast("long") + col(b).cast("long"))

# Final preview
display(final_output.limit(20))
print("Container 12 complete —", final_output.count(), "rows")

In [0]:
# 13 – long/skinny OPRT format (Level preserved)
from pyspark.sql.functions import col, lit, explode, array, struct, when
import pyspark.sql.functions as F

# Use the final_output table from container 12 (has Level + Adj_Org_Code_Final)
df_wide = final_output

# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Adj_Org_Code",
    "All_DNA", "All_First", "All_FU",
    "All_Total", "F2F_Total", "Remote_Total"
]
df_wide = df_wide.drop(*[c for c in cols_to_drop if c in df_wide.columns])

# Step 2: Define identifier columns (KEEP Level)
id_cols = [
    "Der_Activity_Month_Date",
    "Region_Code",
    "ICB",
    "Adj_Org_Code_Final",
    "Treatment_Function_Group",
    "Level",
]

# Step 3: Identify metric columns (exclude identifiers)
metric_cols = [c for c in df_wide.columns if c not in id_cols]

# Helper: safe numeric cast under ANSI mode
# - If value looks like a number → cast to double
# - Else → NULL
def safe_numeric(colname: str):
    return (
        when(
            col(colname).rlike(r'^[-+]?[0-9]*\.?[0-9]+$'),
            col(colname).cast("double")
        )
        .otherwise(F.lit(None).cast("double"))
        .alias("Metric_Value")
    )

# Step 4: Melt into long/skinny format
opa_oprt_long = (
    df_wide.select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("OPRT_Metric_Name"),
                safe_numeric(c)
            )
            for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.OPRT_Metric_Name"),
        col("kv.Metric_Value")
    )
)

display(opa_oprt_long.limit(10))
print(f"Container 13 complete — {opa_oprt_long.count()} rows, {len(opa_oprt_long.columns)} columns")

# Ensure Container 14 uses the cleaned long-format table
final_output = opa_oprt_long




In [0]:
#14 — Join Internal IDs and Clean Long OPRT Dataset (no restacking; Level preserved)
from pyspark.sql.functions import (
    col, lit, concat_ws, lower, regexp_replace, trim
)
from pyspark.sql.types import StringType, DoubleType, DateType
import pyspark.sql.functions as F

# --- Step 1: Start from container 13 output (already has Level + Adj_Org_Code_Final) ---
df_long = final_output

# --- Step 2: DO NOT restack; just keep the dataset as-is ---
df_stacked = df_long

# --- Step 3: Filter out unwanted Treatment_Function_Group = 'Other' ---
df_stacked = df_stacked.filter(col("Treatment_Function_Group") != "Other")

# --- Step 4: Build combined metric name for joining to ID list ---
df_stacked = df_stacked.withColumn(
    "OPRT_Metric_Name_TFC",
    concat_ws("_", col("OPRT_Metric_Name"), col("Treatment_Function_Group"))
)

# --- Step 5: Normalize join keys on our long dataset ---
# spaces -> underscores, drop non [a-z0-9_], collapse underscores, trim underscores, lowercase
df_stacked_clean = df_stacked.withColumn(
    "join_metric",
    lower(regexp_replace(trim(col("OPRT_Metric_Name_TFC")), r"\s+", "_"))
)
df_stacked_clean = df_stacked_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"[^a-z0-9_]", "")
)
df_stacked_clean = df_stacked_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"_+", "_")
)
df_stacked_clean = df_stacked_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"^_+|_+$", "")
)

# --- Step 5b: Prepare mhs_metric_list cleaned keys (bring in Description too) ---
metric_list_col = [c for c in mhs_metric_list.columns if "OPRT" in c and "TFC" in c]
if len(metric_list_col) == 0:
    raise ValueError("Could not find OPRT TFC metric column name in mhs_metric_list. Review mhs_metric_list columns.")
metric_list_col = metric_list_col[0]

mhs_metric_list_clean = mhs_metric_list.withColumn(
    "join_metric",
    lower(regexp_replace(trim(col(metric_list_col)), r"\s+", "_"))
)
mhs_metric_list_clean = mhs_metric_list_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"[^a-z0-9_]", "")
)
mhs_metric_list_clean = mhs_metric_list_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"_+", "_")
)
mhs_metric_list_clean = mhs_metric_list_clean.withColumn(
    "join_metric",
    regexp_replace(col("join_metric"), r"^_+|_+$", "")
)

select_cols = ["join_metric", "InternalID"]
if "Description" in mhs_metric_list_clean.columns:
    select_cols.append("Description")
else:
    print("WARNING: mhs_metric_list does not contain a column named 'Description'.")

mhs_metric_list_clean_sel = mhs_metric_list_clean.select(*select_cols).distinct()

# --- Step 6a: Left join first — to capture unmatched metrics for debugging ---
df_with_id_left = df_stacked_clean.join(
    mhs_metric_list_clean_sel,
    on="join_metric",
    how="left"
)

# Diagnostics (optional)
unmatched = df_with_id_left.filter(col("InternalID").isNull()).select("OPRT_Metric_Name_TFC").distinct()
# If you want to persist unmatched metrics for debugging as Delta:
# dbutils.fs.rm("/mnt/output/unmatched_oprt_metrics", True)  # run once if needed
# unmatched.write.mode("overwrite").format("delta").save("/mnt/output/unmatched_oprt_metrics")

# --- Step 6b: Final join for production (inner; drop unmapped) ---
df_with_id = df_stacked_clean.join(
    mhs_metric_list_clean_sel,
    on="join_metric",
    how="inner"
).drop("join_metric")

# --- Step 7: Filter allowable org codes if provided ---
if "Org_Code" in mhs_allowable_orgs.columns:
    df_with_id = df_with_id.join(
        mhs_allowable_orgs.select(col("Org_Code").alias("allowable_code")),
        df_with_id["Adj_Org_Code_Final"] == col("allowable_code"),
        "inner"
    ).drop("allowable_code")

# --- Step 8: Enforce clean data types and ensure Description present ---
df_with_id = (
    df_with_id
    .withColumn("Der_Activity_Month_Date", col("Der_Activity_Month_Date").cast(DateType()))
    .withColumn("Adj_Org_Code_Final", col("Adj_Org_Code_Final").cast(StringType()))
    .withColumn("Level", col("Level").cast(StringType()))
    .withColumn("Treatment_Function_Group", col("Treatment_Function_Group").cast(StringType()))
    .withColumn("OPRT_Metric_Name", col("OPRT_Metric_Name").cast(StringType()))
    .withColumn("OPRT_Metric_Name_TFC", col("OPRT_Metric_Name_TFC").cast(StringType()))
    .withColumn("Metric_Value", col("Metric_Value").cast(DoubleType()))
)

if "Description" not in df_with_id.columns:
    df_with_id = df_with_id.withColumn("Description", lit(None).cast(StringType()))

# --- Step 9: Write final tidy dataset as Delta on a path ---
# If you previously wrote parquet here, run this ONCE (then comment again):
# dbutils.fs.rm("/mnt/output/opa_oprt_final", True)

#df_with_id.write.mode("overwrite").format("delta").save("/mnt/output/opa_oprt_final")

display(df_with_id.limit(20))
print(f" Container 14 complete — {df_with_id.count()} rows, {len(df_with_id.columns)} columns")


In [0]:
#15 – Add Remote Lower Benchmark
from pyspark.sql import functions as F

# Start from container 10 output
df_benchmark = opa_final_with_added_metrics

# Step 1: Calculate 25th percentile of Remote_Total by month and Adj Org Code
remote_lower = (
    df_benchmark
    .groupBy("Der_Activity_Month_Date", "Adj Org Code")
    .agg(
        F.expr("percentile_approx(Remote_Total, 0.25)").alias("Remote_Lower_Benchmark")
    )
)

# Step 2: Join benchmark back to main dataset
df_with_benchmark = df_benchmark.join(
    remote_lower,
    on=["Der_Activity_Month_Date", "Adj Org Code"],
    how="left"
)

# Step 3: For missing values, fill with 0
df_with_benchmark = df_with_benchmark.fillna({"Remote_Lower_Benchmark": 0})

# Step 4: Final outputs
opa_final_with_remote_benchmark = df_with_benchmark

# Create a standalone lower benchmark table for downstream use
df_lower_bm = (
    remote_lower
    .withColumnRenamed("Adj Org Code", "Adj_Org_Code_Final")
    .select("Der_Activity_Month_Date", "Adj_Org_Code_Final", "Remote_Lower_Benchmark")
)

# Optional: Save for reference or later join, as Delta on a path
# If this path already has parquet, run ONCE:
# dbutils.fs.rm("/mnt/output/opa_lower_benchmark", True)

#df_lower_bm.write.mode("overwrite").format("delta").save("/mnt/output/opa_lower_benchmark")

display(opa_final_with_remote_benchmark.limit(10))
#print(f"Container 15 complete — {opa_final_with_remote_benchmark.count()} rows, {len(opa_final_with_remote_benchmark.columns)} columns")
print(f"Lower benchmark table created — {df_lower_bm.count()} rows, {len(df_lower_bm.columns)} columns")




In [0]:
# 16 — Correct aggregation + early allowable filter + 6-month window + tidy layout (fixed union)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------
# 0) Start from the org-level wide (same source as #12)
# -----------------------------
base0 = opa_final_with_added_metrics.withColumnRenamed("Adj Org Code", "Adj_Org_Code")

# Remove 'Other' TFC like #14
if "Treatment_Function_Group" in base0.columns:
    base0 = base0.filter(F.col("Treatment_Function_Group") != "Other")

# -----------------------------
# 1) EARLY allowable-org filter (from #14, but BEFORE any roll-ups)
# -----------------------------
if "Org_Code" in mhs_allowable_orgs.columns:
    allow = mhs_allowable_orgs.select(F.col("Org_Code").alias("allowable_code")).distinct()
    base0 = base0.join(allow, base0["Adj_Org_Code"] == F.col("allowable_code"), "inner") \
                 .drop("allowable_code")

# -----------------------------
# 2) Re-aggregate to enforce unique grain (month x org x icb x region x TFC)
# -----------------------------
count_cols = [
    "All_Total","All_First","All_FU","All_Proc","All_NoProc",
    "All_FU_Proc","All_FU_NoProc",
    "F2F_Total","F2F_First","F2F_FU","F2F_DNA",
    "Remote_Total","Remote_First","Remote_FU","Remote_DNA",
    "All_DNA","All_First_DNA","All_FU_DNA",
    "All_2WW","All_First_2WW","All_FU_2WW",
    "All_2WW_DNA","All_First_2WW_DNA","All_FU_2WW_DNA"
]
id_org = ["Der_Activity_Month_Date","Adj_Org_Code","ICB","Region_Code","Treatment_Function_Group"]

base = (
    base0.groupBy(*id_org)
         .agg(*[F.sum(F.col(c)).alias(c) for c in count_cols])
)

# -----------------------------
# 3) Recreate the rate/derived columns (same as #12)
# -----------------------------
def add_rate_metrics(df):
    return (
        df
        .withColumn("All_DNA_Over_All_Total",
            F.when((F.col("All_Total")+F.col("All_DNA"))!=0,
                   (F.col("All_DNA")/(F.col("All_Total")+F.col("All_DNA")))*100).otherwise(None))
        .withColumn("All_DNA_Over_All_Total_IG",
            F.when((F.col("All_Total")+F.col("All_DNA"))!=0,
                   (F.col("All_DNA")/(F.col("All_Total")+F.col("All_DNA")))*100).otherwise(None))
        .withColumn("All_First_DNA_Over_All_First",
            F.when((F.col("All_First")+F.col("All_First_DNA"))!=0,
                   (F.col("All_First_DNA")/(F.col("All_First")+F.col("All_First_DNA")))*100).otherwise(None))
        .withColumn("All_First_DNA_Over_All_First_IG",
            F.when((F.col("All_First")+F.col("All_First_DNA"))!=0,
                   (F.col("All_First_DNA")/(F.col("All_First")+F.col("All_First_DNA")))*100).otherwise(None))
        .withColumn("All_FU_DNA_Over_All_FU",
            F.when((F.col("All_FU")+F.col("All_FU_DNA"))!=0,
                   (F.col("All_FU_DNA")/(F.col("All_FU")+F.col("All_FU_DNA")))*100).otherwise(None))
        .withColumn("All_FU_DNA_Over_All_FU_IG",
            F.when((F.col("All_FU")+F.col("All_FU_DNA"))!=0,
                   (F.col("All_FU_DNA")/(F.col("All_FU")+F.col("All_FU_DNA")))*100).otherwise(None))
        .withColumn("All_FU_NoProc_Over_All_FU",
            F.when(F.col("All_FU")!=0,(F.col("All_FU_NoProc")/F.col("All_FU"))*100).otherwise(None))
        .withColumn("All_FU_Proc_Over_All_FU",
            F.when(F.col("All_FU")!=0,(F.col("All_FU_Proc")/F.col("All_FU"))*100).otherwise(None))
        .withColumn("All_FU_To_All_First",
            F.when(F.col("All_First")!=0,(F.col("All_FU")/F.col("All_First"))).otherwise(None))
        .withColumn("All_2WW_DNA_Over_All_2WW",
            F.when(F.col("All_2WW")!=0,(F.col("All_2WW_DNA")/F.col("All_2WW"))*100).otherwise(None))
        .withColumn("All_First_2WW_DNA_Over_All_First_2WW",
            F.when(F.col("All_First_2WW")!=0,(F.col("All_First_2WW_DNA")/F.col("All_First_2WW"))*100).otherwise(None))
        .withColumn("All_FU_2WW_DNA_Over_All_FU_2WW",
            F.when(F.col("All_FU_2WW")!=0,(F.col("All_FU_2WW_DNA")/F.col("All_FU_2WW"))*100).otherwise(None))
        .withColumn("All_FU_Over_All_Total",
            F.when(F.col("All_Total")!=0,(F.col("All_FU")/F.col("All_Total"))*100).otherwise(None))
        .withColumn("All_First_Over_All_Total",
            F.when(F.col("All_Total")!=0,(F.col("All_First")/F.col("All_Total"))*100).otherwise(None))
        .withColumn("All_NoProc_Over_All_Total",
            F.when(F.col("All_Total")!=0,(F.col("All_NoProc")/F.col("All_Total"))*100).otherwise(None))
        .withColumn("All_Proc_Over_All_Total",
            F.when(F.col("All_Total")!=0,(F.col("All_Proc")/F.col("All_Total"))*100).otherwise(None))
        .withColumn("Remote_Total_Over_All_Total",
            F.when(F.col("All_Total")!=0,(F.col("Remote_Total")/F.col("All_Total"))*100).otherwise(None))
        .withColumn("Remote_FU_Over_All_FU",
            F.when(F.col("All_FU")!=0,(F.col("Remote_FU")/F.col("All_FU"))*100).otherwise(None))
        .withColumn("Remote_First_Over_All_First",
            F.when(F.col("All_First")!=0,(F.col("Remote_First")/F.col("All_First"))*100).otherwise(None))
        .withColumn("Remote_DNA_Over_Remote_Total",
            F.when((F.col("Remote_Total")+F.col("Remote_DNA"))!=0,
                   (F.col("Remote_DNA")/(F.col("Remote_Total")+F.col("Remote_DNA")))*100).otherwise(None))
        .withColumn("F2F_DNA_Over_F2F_Total",
            F.when((F.col("F2F_Total")+F.col("F2F_DNA"))!=0,
                   (F.col("F2F_DNA")/(F.col("F2F_Total")+F.col("F2F_DNA")))*100).otherwise(None))
    )

base = add_rate_metrics(base)

# -----------------------------
# 4) Build Org / ICB / Region views from the cleaned org-level base
# -----------------------------
# ORG — keep only Adj_Org_Code_FINAL (drop the raw column so union works)
org_df = (
    base.withColumn("Level", F.lit("Org"))
        .withColumn("Adj_Org_Code_Final", F.col("Adj_Org_Code"))
        .drop("Adj_Org_Code")
)

# ICB
icb_df = (
    base.groupBy("Der_Activity_Month_Date","ICB","Region_Code","Treatment_Function_Group")
        .agg(*[F.sum(F.col(c)).alias(c) for c in count_cols])
        .transform(add_rate_metrics)
        .withColumn("Level", F.lit("ICB"))
        .withColumn("Adj_Org_Code_Final", F.col("ICB"))
)

# REGION
region_df = (
    base.groupBy("Der_Activity_Month_Date","Region_Code","Treatment_Function_Group")
        .agg(*[F.sum(F.col(c)).alias(c) for c in count_cols])
        .transform(add_rate_metrics)
        .withColumn("Level", F.lit("Region"))
        .withColumn("Adj_Org_Code_Final", F.col("Region_Code"))
)

# Align columns for union (now none of the frames has 'Adj_Org_Code')
stacked = (
    org_df.unionByName(icb_df, allowMissingColumns=True)
          .unionByName(region_df, allowMissingColumns=True)
)

# -----------------------------
# 5) 6-month rolling window (require FULL 6 months)
#     – percentiles per Month × Level × TFC
# -----------------------------
st = stacked.withColumn("MonthStart", F.trunc(F.col("Der_Activity_Month_Date"), "month"))
last_complete = F.add_months(F.trunc(F.current_date(), "month"), -1)
st = st.filter(F.col("MonthStart") <= last_complete)

part_cols = ["Level","Adj_Org_Code_Final","Treatment_Function_Group","Region_Code","ICB"]
min_m = st.agg(F.min("MonthStart").alias("m")).first()["m"]
st = st.withColumn("month_idx", F.floor(F.months_between(F.col("MonthStart"), F.lit(min_m))).cast("int"))

w6 = Window.partitionBy(*part_cols).orderBy("month_idx").rangeBetween(-5, 0)

rolled = (
    st.withColumn("rows6", F.count(F.lit(1)).over(w6))
      .withColumn(
          "Avg_All_DNA",
          F.when(F.col("rows6") == 6, F.avg("All_DNA").over(w6))
           .otherwise(F.lit(None).cast("double"))
      )
      .withColumn(
          "Avg_DNA_rate",
          F.when(F.col("rows6") == 6, F.avg("All_DNA_Over_All_Total").over(w6))
           .otherwise(F.lit(None).cast("double"))
      )
)

# National percentiles per Month x Level x TFC
national = (
    rolled.groupBy("MonthStart","Level","Treatment_Function_Group")
          .agg(
              F.expr("percentile_approx(All_DNA_Over_All_Total, 0.5)").alias("National_Median"),
              F.expr("percentile_approx(All_DNA_Over_All_Total, 0.25)").alias("Percentile_25")
          )
)

with_stats = rolled.join(national, ["MonthStart","Level","Treatment_Function_Group"], "left")

# Opportunity rule (only when we have a full 6 months and >0 DNAs on average)
out18 = (
    with_stats.withColumn(
        "DNA_Opportunity_reduction",
        F.when(F.col("Avg_All_DNA").isNull() | (F.col("Avg_All_DNA") <= 0), "No reduction")
         .when(F.col("Avg_DNA_rate") > F.col("National_Median"), "25% reduction")
         .when((F.col("Avg_DNA_rate") <= F.col("National_Median")) & (F.col("Avg_DNA_rate") > F.col("Percentile_25")), "15% reduction")
         .when(F.col("Avg_DNA_rate") <= F.col("Percentile_25"), "10% reduction")
         .otherwise("No reduction")
    )
    .withColumn(
        "DNA_Opportunity",
        F.when(F.col("DNA_Opportunity_reduction")=="25% reduction", F.round(0.25*F.col("Avg_All_DNA")).cast("int"))
         .when(F.col("DNA_Opportunity_reduction")=="15% reduction", F.round(0.15*F.col("Avg_All_DNA")).cast("int"))
         .when(F.col("DNA_Opportunity_reduction")=="10% reduction", F.round(0.10*F.col("Avg_All_DNA")).cast("int"))
         .otherwise(F.lit(0))
    )
    .select(
        "Der_Activity_Month_Date","MonthStart",
        "Level","Adj_Org_Code_Final","Region_Code","ICB","Treatment_Function_Group",
        "All_DNA","All_Total","All_DNA_Over_All_Total",
        "Avg_All_DNA","Avg_DNA_rate","National_Median","Percentile_25",
        "DNA_Opportunity_reduction","DNA_Opportunity"
    )
    .orderBy("MonthStart","Level","Adj_Org_Code_Final","Treatment_Function_Group")
)

display(out18.limit(50))

# Save DNA opportunity as Delta on a path
# If this path has old parquet from earlier runs, run ONCE:
# dbutils.fs.rm("/mnt/output/opa_dna_opportunity_clean", True)

#out18.write.mode("overwrite").format("delta").save("/mnt/output/opa_dna_opportunity_clean")



In [0]:
#17 Saving the complete file to the lake mart for QA
df_with_id.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/OP_QA_new_metrics_final_UC1b.csv")

#display(df_with_id.limit(10))
#print(f"Number of rows in df_with_id: {df_with_id.count()}")




In [0]:
# Filter for the specific month/date
df_with_id = df_with_id.withColumn("Der_Activity_Month_Date", to_date("Der_Activity_Month_Date", "dd/MM/yyyy"))
# Filter for the specific month/date
df_filtered = df_with_id.filter(df_with_id["Der_Activity_Month_Date"] == to_date(lit("30/09/2025"), "dd/MM/yyyy"))


# Saving the filtered file to the lake mart for QA
df_filtered.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/OP_QA_long_UC1_30092025.csv")




In [0]:
#18 lower_bench mark, saving the file to the lake mart for QA 
opa_final_with_remote_benchmark.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/lower_bm_test_final_UC1.csv")

#display(opa_final_with_remote_benchmark.limit(10))
#print(f"Number of rows in opa_final_with_remote_benchmark: {opa_final_with_remote_benchmark.count()}")

In [0]:
#19 DNA opportunities
out18.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/DNA_Opportunities_test_6_final_UC1.csv")



In [0]:
#20 MHS ingestion
# import sys
# sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')
# from mhs_import import MHS_IngestionHub
# from mhs_db_config import INSERT, DELETE

# def main():
#     max_date = df_with_id.agg(F.max("reportingDate")).collect()[0][0]
#     desc = f"df_with_id_{max_date}"
#     InjectionHub(df_with_id, desc, True)
#     print("df_with_id sent to InjectionHub")

# def InjectionHub(dfih, desc, tf):
#     sdf = dfih.withColumn('reportingDate', F.to_date('reportingDate'))
#     display(sdf.sample(False, 0.01))
#     display(sdf.dtypes)
#     MHS_IngestionHub.upload(
#         mhs_df=sdf,
#         description=desc,
#         loaded_by="steven.evans4@nhs.net",
#         mhs_mode=INSERT,
#         skip_existing_data_check=tf
#     )

# if __name__ == '__main__':
#     main()